# Global Feature Selection (Boruta-SHAP)

This notebook runs the modernized 5-stage feature selection engine on the aggregated global master snapshots. It drops the dimensionality-reduction pre-screening (Stage 1) from the API-level notebooks because we only have a few hundred candidates left. Instead, it upgrades the Boruta step to utilize TreeSHAP values, providing superior noise-filtration against cross-source collinearity.

**Pipeline Stages:**
1. **Boruta-SHAP**: Combines Boruta's shadow-feature noise thresholds with SHAP's mathematically consistent attribution.
2. **Vintage Stability**: Tests cross-regime resilience by fitting models at multiple historical cutoff points (2010, 2014, 2018, etc.).
3. **Cluster Redundancy Check**: NaN-aware Spearman hierarchical clustering to eliminate collinear redundancy *across* APIs (e.g., Unifier ISM vs FRED Manufacturing).
4. **Interaction Rescue**: Recovers features that were discarded but exhibit synergistic non-linear interactions.
5. **Sequential Forward Selection**: Iteratively scores features by walk-forward CV MAE, finalizing the mathematically optimal subset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import re
import json
import lightgbm as lgb
from pathlib import Path
import sys
import warnings
from tqdm.notebook import tqdm
import gc
from collections import Counter
from scipy.cluster import hierarchy
from scipy.stats import spearmanr, binomtest
from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import TimeSeriesSplit

warnings.filterwarnings('ignore')

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from settings import DATA_PATH
from Train.data_loader import load_target_data, pivot_snapshot_to_wide, batch_lagged_target_features
from Train.config import load_selected_features
from Train.feature_engineering import get_calendar_features_dict
from utils.transforms import winsorize_covid_period

SEED = 42
np.random.seed(SEED)

LGB_PARAMS = {
    'objective': 'regression', 'metric': 'l2', 'n_estimators': 100,
    'learning_rate': 0.05, 'num_leaves': 31, 'verbose': -1,
    'n_jobs': 1, 'random_state': SEED,
}

In [ ]:
# === LightGBM Helpers ===
def _sanitize_col(name):
    return re.sub(r'[^A-Za-z0-9_]', '_', str(name))

def _sanitize_df(df):
    clean_to_orig = {}
    new_cols = []
    for c in df.columns:
        clean = _sanitize_col(c)
        base = clean
        i = 2
        while clean in clean_to_orig:
            clean = f"{base}_{i}"
            i += 1
        clean_to_orig[clean] = c
        new_cols.append(clean)
    out = df.copy()
    out.columns = new_cols
    return out, clean_to_orig

def _lgb_fit(X, y, params=None):
    if params is None:
        params = LGB_PARAMS
    X_clean, mapping = _sanitize_df(X)
    model = lgb.LGBMRegressor(**params)
    model.fit(X_clean, y)
    return model, mapping

def _lgb_predict(model, X, mapping):
    orig_to_clean = {v: k for k, v in mapping.items()}
    X_clean = X.rename(columns=orig_to_clean)
    return model.predict(X_clean)

def _lgb_importances(model, mapping):
    clean_names = model.feature_name_
    imp_vals = model.feature_importances_
    return pd.Series({mapping.get(c, c): v for c, v in zip(clean_names, imp_vals)})

In [ ]:
# === 1. Boruta-SHAP ===
def get_boruta_shap_importance(X, y, n_runs=100, alpha=0.05):
    common = X.index.intersection(y.index)
    X_curr = X.loc[common]
    y_curr = y.loc[common]

    feature_names = X_curr.columns.tolist()
    hits = np.zeros(len(feature_names), dtype=int)

    for run in tqdm(range(n_runs), desc="Boruta-SHAP Runs", leave=False):
        rng = np.random.RandomState(SEED + run)
        shadow_data = {}
        for col in feature_names:
            col_vals = X_curr[col].values.copy()
            valid_mask = ~np.isnan(col_vals) if np.issubdtype(col_vals.dtype, np.floating) else np.ones(len(col_vals), dtype=bool)
            valid_vals = col_vals[valid_mask].copy()
            rng.shuffle(valid_vals)
            col_vals[valid_mask] = valid_vals
            shadow_data[f'_shadow_{col}'] = col_vals

        shadow_df = pd.DataFrame(shadow_data, index=X_curr.index)
        X_combined = pd.concat([X_curr, shadow_df], axis=1)

        model, mapping = _lgb_fit(X_combined, y_curr)
        orig_to_clean = {v: k for k, v in mapping.items()}
        X_clean = X_combined.rename(columns=orig_to_clean)
        
        explainer = shap.TreeExplainer(model)
        shap_vals = explainer.shap_values(X_clean)
        
        # SHAP importance is mean absolute SHAP value
        mean_abs_shap = np.abs(shap_vals).mean(axis=0)
        shap_importances = pd.Series(mean_abs_shap, index=X_combined.columns)

        shadow_cols = [c for c in X_combined.columns if c.startswith('_shadow_')]
        shadow_threshold = np.percentile(shap_importances[shadow_cols].values, 75)

        for i, feat in enumerate(feature_names):
            if shap_importances.get(feat, 0) > shadow_threshold:
                hits[i] += 1

    confirmed, tentative = [], []
    for i, feat in enumerate(feature_names):
        p_val = binomtest(hits[i], n=n_runs, p=0.25, alternative='greater').pvalue
        if p_val < alpha:
            confirmed.append(feat)
        elif p_val < alpha * 5:
            tentative.append(feat)

    print(f"   Boruta-SHAP: {len(confirmed)} confirmed, {len(tentative)} tentative "
          f"(hit rates: min={hits.min()}/{n_runs}, max={hits.max()}/{n_runs})")
    return confirmed + tentative


In [ ]:
# === 2. Vintage Stability ===
def get_vintage_stability(X, y, feature_list, min_presence=2):
    years = ['2010', '2014', '2018', '2022', 'Latest']
    weights = {'2010': 1, '2014': 2, '2018': 4, '2022': 8, 'Latest': 16}
    scores = pd.DataFrame(np.nan, index=feature_list, columns=years)

    for year in years:
        if year == 'Latest':
            cutoff = X.index.max()
        else:
            cutoff = pd.to_datetime(f"{year}-12-01")

        X_vintage = X[X.index <= cutoff][feature_list].copy()
        y_vintage = y[y.index <= cutoff].copy()
        if len(X_vintage) < 50:
            continue

        X_vintage = winsorize_covid_period(X_vintage)
        common = X_vintage.index.intersection(y_vintage.index)
        if len(common) < 50:
            continue

        model, mapping = _lgb_fit(X_vintage.loc[common], y_vintage.loc[common])
        imp = _lgb_importances(model, mapping)
        if imp.sum() > 0:
            imp = imp / imp.sum()
            for feat in imp.index:
                if feat in scores.index:
                    scores.loc[feat, year] = imp[feat]

    weight_series = pd.Series(weights)
    results = {}
    for feat in feature_list:
        feat_scores = scores.loc[feat].dropna()
        if len(feat_scores) == 0:
            continue
        if (feat_scores > 0).sum() < min_presence:
            continue
        available_weights = weight_series[feat_scores.index]
        weighted_score = (feat_scores * available_weights).sum() / available_weights.sum()
        results[feat] = weighted_score

    result_series = pd.Series(results)
    latest_scores = scores['Latest'].dropna()
    latest_nonzero = latest_scores[latest_scores > 0].index
    return result_series[result_series.index.isin(latest_nonzero)].sort_values(ascending=False)

In [ ]:
# === 3. Cluster Redundancy ===
def cluster_redundancy(X, feature_list, target_series, max_clusters=50, min_overlap=30):
    if len(feature_list) <= max_clusters:
        return feature_list
    X_curr = X[feature_list].copy()
    common = X_curr.index.intersection(target_series.index)
    X_curr = X_curr.loc[common]
    y_curr = target_series.loc[common]
    valid_target = y_curr.notna()
    X_curr = X_curr.loc[valid_target]
    y_curr = y_curr.loc[valid_target]

    corr = X_curr.corr(method='spearman').fillna(0)
    for i, col_i in enumerate(feature_list):
        for j, col_j in enumerate(feature_list):
            if i >= j:
                continue
            if (X_curr[col_i].notna() & X_curr[col_j].notna()).sum() < min_overlap:
                corr.loc[col_i, col_j] = 0
                corr.loc[col_j, col_i] = 0

    dist_matrix = 1 - corr.abs().values
    np.fill_diagonal(dist_matrix, 0)
    dist_matrix = (np.maximum(dist_matrix, 0) + np.maximum(dist_matrix, 0).T) / 2

    from scipy.spatial.distance import squareform
    linkage = hierarchy.ward(squareform(dist_matrix, checks=False))
    cluster_ids = hierarchy.fcluster(linkage, t=max_clusters, criterion='maxclust')

    X_imputed = X_curr.fillna(X_curr.median())
    mi = mutual_info_regression(X_imputed.fillna(0), y_curr, random_state=SEED)
    mi_series = pd.Series(mi, index=feature_list)

    selected = []
    for cluster_id in np.unique(cluster_ids):
        members = [feature_list[i] for i, c in enumerate(cluster_ids) if c == cluster_id]
        selected.append(mi_series[members].idxmax())
    return selected

In [ ]:
# === 4. Interaction Rescue ===
def _cv_mae_helper(feature_set, X, y, tscv):
    if not feature_set:
        return float('inf')
    X_sub = X[feature_set]
    maes = []
    for train_idx, test_idx in tscv.split(X_sub):
        model, mapping = _lgb_fit(X_sub.iloc[train_idx], y.iloc[train_idx])
        preds = _lgb_predict(model, X_sub.iloc[test_idx], mapping)
        maes.append(np.mean(np.abs(y.iloc[test_idx].values - preds)))
    return np.mean(maes)

def interaction_rescue(X, y, confirmed_features, rejected_pool, n_splits=8, gap=3, top_k=10):
    common = X.index.intersection(y.index)
    X_curr, y_curr = X.loc[common], y.loc[common]
    rejected = [f for f in rejected_pool if f in X_curr.columns and f not in confirmed_features]
    if not rejected:
        return []

    tscv = TimeSeriesSplit(n_splits=n_splits, gap=gap)
    baseline_mae = _cv_mae_helper(confirmed_features, X_curr, y_curr, tscv)

    single_improvements = {}
    for feat in tqdm(rejected, desc="Rescue: Single-feature scan", leave=False):
        trial_mae = _cv_mae_helper(confirmed_features + [feat], X_curr, y_curr, tscv)
        imp = (baseline_mae - trial_mae) / (baseline_mae + 1e-12)
        if imp > 0:
            single_improvements[feat] = imp

    single_rescued = sorted(single_improvements.items(), key=lambda x: x[1], reverse=True)[:top_k]
    for feat, imp in single_rescued:
        print(f"   Rescued (single): {feat} (delta MAE = {imp:.4f})")

    def _extract_split_pairs(model, mapping):
        pair_counts = Counter()
        clean_names = model.feature_name_
        booster = model.booster_
        for tree_idx in range(booster.num_trees()):
            def _walk(node, parent_feature=None):
                if 'split_feature' not in node: return
                feat_idx = node['split_feature']
                feat_name = mapping.get(clean_names[feat_idx], f'feature_{feat_idx}') if feat_idx < len(clean_names) else f'feature_{feat_idx}'
                if parent_feature is not None and parent_feature != feat_name:
                    pair_counts[tuple(sorted([parent_feature, feat_name]))] += 1
                if 'left_child' in node: _walk(node['left_child'], feat_name)
                if 'right_child' in node: _walk(node['right_child'], feat_name)
            _walk(booster.dump_model()['tree_info'][tree_idx]['tree_structure'])
        return pair_counts

    model_full, mapping_full = _lgb_fit(X_curr, y_curr, params={**LGB_PARAMS, 'n_estimators': 200, 'num_leaves': 63, 'max_depth': 6})
    pair_counts = _extract_split_pairs(model_full, mapping_full)

    rejected_set = set(rejected)
    pair_rescued = []
    for (feat_a, feat_b), count in pair_counts.most_common():
        if len(pair_rescued) >= top_k: break
        new_feats = [f for f in (feat_a, feat_b) if f in rejected_set and f not in [x[0] for x in single_rescued]]
        if not new_feats: continue

        base = list(dict.fromkeys(confirmed_features + [f for f, _ in single_rescued]))
        trial = list(dict.fromkeys(base + new_feats))
        
        base_mae = _cv_mae_helper(base, X_curr, y_curr, tscv)
        trial_mae = _cv_mae_helper(trial, X_curr, y_curr, tscv)
        imp = (base_mae - trial_mae) / (base_mae + 1e-12)
        if imp > 0:
            for f in new_feats:
                if f not in [pf for pf, _ in pair_rescued]:
                    pair_rescued.append((f, imp))
                    print(f"   Rescued (pair w/ {feat_a if f!=feat_a else feat_b}): {f} (delta = {imp:.4f})")

    all_rescued = [f for f, _ in single_rescued] + [f for f, _ in pair_rescued]
    return list(dict.fromkeys(all_rescued))

In [ ]:
# === 5. Sequential Forward Selection ===
def sequential_forward_selection(X, y, candidates, n_splits=8, gap=3, min_improvement=0.001, patience=3, min_features=5):
    if len(candidates) == 0:
        return []
    common = X.index.intersection(y.index)
    X_curr, y_curr = X.loc[common], y.loc[common]
    tscv = TimeSeriesSplit(n_splits=n_splits, gap=gap)

    best_single_mae, best_single = float('inf'), candidates[0]
    for feat in candidates[:min(20, len(candidates))]:
        mae = _cv_mae_helper([feat], X_curr, y_curr, tscv)
        if mae < best_single_mae:
            best_single_mae, best_single = mae, feat

    selected = [best_single]
    current_mae = best_single_mae
    remaining = [f for f in candidates if f != best_single]
    fails = 0
    print(f"   SFS start: {best_single} (MAE={current_mae:.2f})")

    while remaining:
        best_next, best_next_mae = None, current_mae
        for feat in remaining:
            mae = _cv_mae_helper(selected + [feat], X_curr, y_curr, tscv)
            if mae < best_next_mae:
                best_next_mae, best_next = mae, feat

        if best_next is None:
            fails += 1
            if len(selected) >= min_features and fails >= patience:
                print(f"   SFS stop: no improvement for {patience} rounds")
                break
            forced = remaining[0]
            selected.append(forced); remaining.remove(forced)
            print(f"   SFS +{forced} (forced, below min_features)")
            continue

        imp = (current_mae - best_next_mae) / (current_mae + 1e-12)
        if imp < min_improvement:
            fails += 1
            if len(selected) >= min_features and fails >= patience:
                print(f"   SFS stop: improvements < {min_improvement}")
                break
        else:
            fails = 0
            
        selected.append(best_next)
        remaining.remove(best_next)
        current_mae = best_next_mae
        print(f"   SFS +{best_next} (MAE={current_mae:.2f}, delta={imp:.4f})")

    print(f"   SFS final: {len(selected)} features, MAE={current_mae:.2f}")
    return selected

In [ ]:
# === Construct Dataset ===
print("Loading target data...")
target_nsa = load_target_data('nsa', 'first', 'revised', use_cache=False)
target_sa = load_target_data('sa', 'first', 'revised', use_cache=False)

lags_nsa = batch_lagged_target_features(target_nsa, prefix='nfp_nsa')
lags_sa = batch_lagged_target_features(target_sa, prefix='nfp_sa')

print("Loading available pre-selected features across all API sources...")
selected_features_nsa = load_selected_features('nsa')
selected_features_sa = load_selected_features('sa')

def build_X(target_df, selected_features, lags_lookup):
    X_list, months = [], []
    snapshots_dir = DATA_PATH / "Exogenous_data" / "master_snapshots" / "decades"
    
    for _, row in tqdm(target_df.dropna(subset=['y_mom']).iterrows(), desc="Building X matrix", total=len(target_df.dropna(subset=['y_mom']))):
        tm = row['ds']
        decade = f"{tm.year // 10 * 10}s"
        snap_path = snapshots_dir / decade / str(tm.year) / f"{tm.strftime('%Y-%m')}.parquet"
        if not snap_path.exists():
            continue
            
        snap_df = pd.read_parquet(snap_path)
        wide_df = pivot_snapshot_to_wide(snap_df, selected_features)
        if wide_df.empty:
            continue
            
        feats = wide_df.iloc[0].to_dict()
        if tm in lags_lookup:
            feats.update(lags_lookup[tm])
        feats.update(get_calendar_features_dict(tm))
        
        X_list.append(feats)
        months.append(tm)
        
    X = pd.DataFrame(X_list, index=pd.DatetimeIndex(months, name='date'))
    X = X.dropna(axis=1, how='all')
    return X

print("Building NSA X matrix...")
X_nsa = build_X(target_nsa, selected_features_nsa, lags_nsa)
print(f"NSA X matrix built: {X_nsa.shape}")

print("Building SA X matrix...")
X_sa = build_X(target_sa, selected_features_sa, lags_sa)
print(f"SA X matrix built: {X_sa.shape}")

In [ ]:
# === Run 5-Stage Global Pipeline ===
target_nsa['y_acc'] = target_nsa['y_mom'].diff()
target_sa['y_acc'] = target_sa['y_mom'].diff()
target_nsa_series = target_nsa.dropna(subset=['y_mom']).set_index('ds')
target_sa_series = target_sa.dropna(subset=['y_mom']).set_index('ds')

targets = {
    'NSA': (X_nsa, target_nsa_series['y_mom']),
    'SA': (X_sa, target_sa_series['y_mom']),
    'NSA_Acc': (X_nsa, target_nsa_series['y_acc'].dropna()),
    'SA_Acc': (X_sa, target_sa_series['y_acc'].dropna()),
}

final_global_results = {}

for t_name, (X_mat, y_series) in targets.items():
    print(f"\n{'='*60}")
    print(f"  GLOBAL PIPELINE START: {t_name}")
    print(f"{'='*60}")

    # Match indices
    common = X_mat.index.intersection(y_series.index)
    if len(common) < 50:
        print(f"Not enough overlapping data for {t_name}. Skipping.")
        continue
    
    y_series_matched = y_series.loc[common]
    X_mat_matched = X_mat.loc[common]

    y_series_wins = winsorize_covid_period(y_series_matched)
    X_mat_wins = winsorize_covid_period(X_mat_matched)
    
    all_candidates = X_mat_wins.columns.tolist()
    
    print(f"1. Boruta-SHAP Feature Selection on {len(all_candidates)} features...")
    stage1_candidates = get_boruta_shap_importance(X_mat_wins, y_series_wins, n_runs=100)
    if not stage1_candidates:
        print("No features survived Boruta-SHAP. Falling back to top 30 by absolute correlation.")
        corrs = X_mat_wins.corrwith(y_series_wins).abs()
        stage1_candidates = corrs.nlargest(30).index.tolist()
    print(f"   > Stage 1 Survivors: {len(stage1_candidates)}")

    print("2. Vintage Stability...")
    weighted_scores = get_vintage_stability(X_mat_wins, y_series_wins, stage1_candidates, min_presence=1)
    if len(weighted_scores) > 0:
        stage2_candidates = weighted_scores[weighted_scores > weighted_scores.median() * 0.5].index.tolist()
    else:
        stage2_candidates = stage1_candidates
    print(f"   > Stage 2 Survivors: {len(stage2_candidates)}")

    print("3. Cluster Redundancy Check...")
    max_clust = min(40, len(stage2_candidates)) if len(stage2_candidates) > 0 else 40
    stage3_candidates = cluster_redundancy(X_mat_wins, stage2_candidates, y_series_wins, max_clusters=max_clust)
    print(f"   > Stage 3 Survivors: {len(stage3_candidates)}")

    print("4. Interaction Rescue...")
    rescued = interaction_rescue(X_mat_wins, y_series_wins, stage3_candidates, all_candidates, top_k=5)
    stage4_candidates = list(dict.fromkeys(stage3_candidates + rescued))
    print(f"   > Stage 4 Candidates: {len(stage4_candidates)} ({len(stage3_candidates)} + {len(rescued)} rescued)")
    
    print("5. Sequential Forward Selection...")
    final_list = sequential_forward_selection(
        X_mat_wins, y_series_wins, stage4_candidates,
        n_splits=8, gap=3, min_improvement=0.001, patience=3, min_features=5
    )
    final_global_results[t_name] = final_list
    
    print(f"\nSelected global features ({t_name}):")
    for i, f in enumerate(final_list):
        print(f"  {i+1:2d}. {f}")


In [ ]:
# === Save Final Global Features ===
print(f"\n{'='*60}")
print("Merging MoM + Acceleration features (union) and saving...")

for variant in ['nsa', 'sa']:
    mom_key = variant.upper()
    acc_key = f"{variant.upper()}_Acc"

    mom_feats = set(final_global_results.get(mom_key, []))
    acc_feats = set(final_global_results.get(acc_key, []))
    union_feats = sorted(list(mom_feats | acc_feats))

    print(f"\n{variant.upper()}: {len(mom_feats)} MoM + {len(acc_feats)} Acc -> {len(union_feats)} union")
    
    fname = f"FINAL_selected_features_{variant}.json"
    save_path = PROJECT_ROOT / 'notebooks' / fname
    with open(save_path, 'w') as f:
        json.dump(union_feats, f, indent=4)
    print(f"Saved {save_path.name}")

print("\nDone!")